# AI Stock Predictor: Advanced Calibrated Dashboard

This is the **highly calibrated** version of the trading system, featuring advanced feature engineering, dual-model ensemble logic, and precise signal timing.

## Advanced Enhancements
* **Market Regime Detection:** Uses Volatility Z-Scores and ADX to filter signals based on market conditions.
* **Volume Profile Analysis:** Incorporates VWAP deviation and Volume Z-Scores to identify institutional activity.
* **Probability Calibration:** Optimized XGBoost parameters for better-behaved probability scores.
* **Precise Timing:** Every signal is timestamped with the exact generation time.
* **Risk-Adjusted Ranking:** Trades are ranked by a combination of Confidence and Expected Return.


## 1. Environment Setup


In [5]:
import yfinance as yf
import pandas as pd
import numpy as np
import xgboost as xgb
from datetime import datetime
import pytz
import warnings

warnings.filterwarnings('ignore')
np.random.seed(42)
print(f"Environment setup complete. Current Time: {datetime.now(pytz.timezone('US/Eastern')).strftime('%Y-%m-%d %H:%M:%S')} EST")


Environment setup complete. Current Time: 2026-06-01 11:41:07 EST


## 2. Advanced Data Pipeline


In [6]:
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime

def fetch_data_5y_v3(tickers):
    data = {}
    for ticker in tickers:
        try:
            df = yf.download(ticker, period='5y', progress=False)
            if not df.empty:
                if isinstance(df.columns, pd.MultiIndex):
                    df.columns = df.columns.get_level_values(0)
                data[ticker] = df
        except Exception:
            continue
    return data

def add_advanced_features(df):
    # 1. Basic Indicators
    df['RSI_14'] = (lambda x, w: 100 - (100 / (1 + (x.diff().where(x.diff() > 0, 0).rolling(w).mean() / -x.diff().where(x.diff() < 0, 0).rolling(w).mean()))))(df['Close'], 14)
    df['SMA_20'] = df['Close'].rolling(20).mean()
    df['SMA_50'] = df['Close'].rolling(50).mean()
    
    # 2. Volatility & Regime
    tr = pd.concat([df['High']-df['Low'], abs(df['High']-df['Close'].shift(1)), abs(df['Low']-df['Close'].shift(1))], axis=1).max(axis=1)
    df['ATR_14'] = tr.rolling(14).mean()
    df['Vol_ZScore'] = (df['ATR_14'] - df['ATR_14'].rolling(20).mean()) / df['ATR_14'].rolling(20).std()
    
    # 3. Volume Profile
    df['Typical_Price'] = (df['High'] + df['Low'] + df['Close']) / 3
    df['VWAP'] = (df['Typical_Price'] * df['Volume']).cumsum() / df['Volume'].cumsum()
    df['VWAP_Dist'] = (df['Close'] - df['VWAP']) / df['VWAP']
    df['Volume_ZScore'] = (df['Volume'] - df['Volume'].rolling(20).mean()) / df['Volume'].rolling(20).std()
    
    # 4. Trend Strength (Simplified ADX)
    up_move = df['High'] - df['High'].shift(1)
    down_move = df['Low'].shift(1) - df['Low']
    plus_dm = np.where((up_move > down_move) & (up_move > 0), up_move, 0)
    minus_dm = np.where((down_move > up_move) & (down_move > 0), down_move, 0)
    df['Plus_DI'] = 100 * (pd.Series(plus_dm).rolling(14).mean().values / df['ATR_14'].values)
    df['Minus_DI'] = 100 * (pd.Series(minus_dm).rolling(14).mean().values / df['ATR_14'].values)
    df['ADX'] = 100 * (abs(df['Plus_DI'] - df['Minus_DI']) / (df['Plus_DI'] + df['Minus_DI'])).rolling(14).mean()
    
    # 5. Time Features
    df['Day_of_Week'] = df.index.dayofweek
    
    return df

def generate_labels_v3(df, threshold=0.005):
    df['Next_Open'] = df['Open'].shift(-1)
    df['Next_Close'] = df['Close'].shift(-1)
    df['Next_Day_Return'] = (df['Next_Close'] - df['Next_Open']) / df['Next_Open']
    df['Label'] = np.select([(df['Next_Day_Return'] > threshold), (df['Next_Day_Return'] < -threshold)], [2, 1], default=0)
    return df



## 3. Calibrated Signal Generation


In [7]:
def get_calibrated_recommendations(tickers, threshold=0.005, confidence_cutoff=0.55):
    results = []
    print(f"Processing {len(tickers)} tickers with Advanced Calibration...")
    
    for ticker in tickers:
        try:
            # Fetch and Process
            df = yf.download(ticker, period='5y', progress=False)
            if df.empty: continue
            if isinstance(df.columns, pd.MultiIndex): df.columns = df.columns.get_level_values(0)
            
            df = add_advanced_features(df)
            df = generate_labels_v3(df, threshold)
            
            # Feature Selection
            exclude = ['Open', 'High', 'Low', 'Close', 'Volume', 'Adj Close', 'Next_Open', 'Next_Close', 'Next_Day_Return', 'Label', 'Typical_Price', 'VWAP']
            features = [c for c in df.columns if c not in exclude]
            
            train_df = df.dropna(subset=['Label', 'ADX', 'Vol_ZScore']).iloc[:-1]
            if len(train_df) < 150: continue
            
            X_train = train_df[features].values
            y_class = train_df['Label'].values
            y_reg = train_df['Next_Day_Return'].values
            
            current_state = df[features].iloc[-1:].values
            last_price = df['Close'].iloc[-1]
            
            # 1. Calibrated Classifier
            clf = xgb.XGBClassifier(
                n_estimators=150, max_depth=4, learning_rate=0.05,
                subsample=0.8, colsample_bytree=0.8,
                objective='multi:softprob', num_class=3, verbosity=0
            )
            clf.fit(X_train, y_class)
            
            # 2. Robust Regressor
            reg = xgb.XGBRegressor(n_estimators=150, max_depth=4, learning_rate=0.05, subsample=0.8, verbosity=0)
            reg.fit(X_train, y_reg)
            
            probs = clf.predict_proba(current_state)[0]
            exp_ret = reg.predict(current_state)[0]
            
            pred_class = np.argmax(probs)
            conf = probs[pred_class]
            
            action = "HOLD"
            if conf >= confidence_cutoff:
                if pred_class == 2: action = "BUY"
                elif pred_class == 1: action = "SELL/SHORT"
            
            results.append({
                'Ticker': ticker,
                'Action': action,
                'Confidence': conf,
                'Exp_Return_%': exp_ret * 100,
                'Exp_Dollar_Move': last_price * exp_ret,
                'Price': last_price,
                'Prob_Up': probs[2],
                'Prob_Down': probs[1],
                'ADX': df['ADX'].iloc[-1],
                'Vol_Regime': "High" if df['Vol_ZScore'].iloc[-1] > 1 else "Normal"
            })
        except Exception: continue
        
    return pd.DataFrame(results)


## 4. Final Calibrated Dashboard


In [8]:
# Ticker List
my_tickers = ['AA', 'AAL', 'AAP', 'ABBV', 'ABNB', 'ABT', 'ACN', 'ADBE', 'ADI', 'ADM', 'ADP', 'AEP', 'AES', 'AFL', 'AIG', 'AIZ', 'AJG', 'AKAM', 'ALB', 'ALLE', 'ALL', 'ALNY', 'AMAT', 'AMD', 'AMCR', 'AME', 'AMP', 'AMZN', 'ANET', 'AON', 'APA', 'APD', 'APH', 'APTV', 'ARE', 'ARM', 'ASML', 'ATO', 'AVB', 'AVGO', 'AVY', 'AWK', 'AXON', 'AXP', 'AZN', 'BA', 'BABA', 'BAC', 'BDX', 'BEN', 'BF-B', 'BIIB', 'BIO', 'BK', 'BKNG', 'BKR', 'BLK', 'BMY', 'BRK-B', 'BSX', 'C', 'CAG', 'CAH', 'CAT', 'CB', 'CBRE', 'CCI', 'CDNS', 'CE', 'CF', 'CFG', 'CHD', 'CHTR', 'CI', 'CINF', 'CL', 'CLX', 'CMCSA', 'CME', 'CMG', 'CNC', 'CNP', 'COF', 'COIN', 'COO', 'COP', 'COST', 'CPB', 'CPRT', 'CRL', 'CRM', 'CRWD', 'CSCO', 'CSX', 'CTAS', 'CTRA', 'CTSH', 'CVS', 'CVX', 'D', 'DAL', 'DDOG', 'DE', 'DELL', 'DG', 'DHR', 'DIA', 'DIS', 'DLTR', 'DOCU', 'DOV', 'DRI', 'DTE', 'DUK', 'DVN', 'DXCM', 'EA', 'EBAY', 'ECL', 'ED', 'EFX', 'EIX', 'EL', 'EMR', 'ENPH', 'EOG', 'EPAM', 'EQIX', 'ESS', 'ETN', 'ETSY', 'EVRG', 'EXC', 'EXPE', 'F', 'FANG', 'FAST', 'FDX', 'FE', 'FIS', 'FISV', 'FITB', 'FMC', 'FOX', 'FOXA', 'FRT', 'FTNT', 'GD', 'GE', 'GEN', 'GILD', 'GIS', 'GL', 'GLW', 'GM', 'GNRC', 'GOOG', 'GOOGL', 'GS', 'HAL', 'HAS', 'HBAN', 'HCA', 'HD', 'HIG', 'HLT', 'HOLX', 'HON', 'HPE', 'HPQ', 'HRL', 'HST', 'HSY', 'HUM', 'HWM', 'IBM', 'ICE', 'ILMN', 'INTC', 'INTU', 'IP', 'IR', 'ISRG', 'ITW', 'IWM', 'JBHT', 'JCI', 'JD', 'JKHY', 'JNJ', 'JPM', 'KEY', 'KHC', 'KIM', 'KLAC', 'KMB', 'KMI', 'KO', 'KR', 'LEN', 'LH', 'LIN', 'LKQ', 'LLY', 'LMT', 'LNT', 'LOW', 'LRCX', 'LULU', 'LVS', 'LW', 'LYB', 'MA', 'MAA', 'MAR', 'MAS', 'MCD', 'MCHP', 'MCO', 'MDLZ', 'MDT', 'MELI', 'META', 'MET', 'MGM', 'MKC', 'MLM', 'MMM', 'MMC', 'MNST', 'MO', 'MOS', 'MPWR', 'MRK', 'MRNA', 'MRVL', 'MS', 'MSCI', 'MSFT', 'MSTR', 'MTB', 'MU', 'MUFG', 'NDAQ', 'NEE', 'NEM', 'NET', 'NFLX', 'NI', 'NKE', 'NOC', 'NOW', 'NRG', 'NTES', 'NTNX', 'NTRS', 'NVDA', 'NXPI', 'ODFL', 'OKTA', 'OMC', 'ON', 'ORCL', 'ORLY', 'OTIS', 'OXY', 'PANW', 'PAYC', 'PAYX', 'PCAR', 'PDD', 'PEG', 'PEP', 'PFE', 'PFG', 'PG', 'PH', 'PKG', 'PLTR', 'PM', 'PNC', 'PPG', 'PRU', 'PSA', 'PSX', 'PWR', 'PYPL', 'QCOM', 'QQQ', 'REGN', 'RF', 'RJF', 'RIVN', 'RL', 'RMD', 'ROKU', 'ROP', 'ROST', 'RSG', 'RTX', 'SBUX', 'SCHW', 'SEDG', 'SHOP', 'SJM', 'SLB', 'SMCI', 'SNOW', 'SNPS', 'SO', 'SOFI', 'SPGI', 'SPY', 'SQ', 'STE', 'STT', 'STX', 'SWK', 'SWKS', 'SYF', 'SYK', 'T', 'TEL', 'TER', 'TFC', 'TJX', 'TMO', 'TMUS', 'TROW', 'TRV', 'TSCO', 'TSLA', 'TSM', 'TT', 'TTD', 'TXN', 'TXT', 'UAL', 'UBER', 'UDR', 'UHS', 'UNH', 'UNP', 'USB', 'V', 'VEEV', 'VICI', 'VLO', 'VMC', 'VRSK', 'VRSN', 'VRTX', 'VZ', 'WAB', 'WBA', 'WDAY', 'WEC', 'WFC', 'WHR', 'WM', 'WMT', 'WRB', 'WSM', 'XYL', 'YUM', 'ZBH', 'ZM', 'ZS']
# Run Analysis
recs = get_calibrated_recommendations(my_tickers)

# Precise Timestamp
est = pytz.timezone('US/Eastern')
now = datetime.now(est)

print("\n" + "="*80)
print(f"       STOCK PREDICTOR: CALIBRATED SIGNALS | {now.strftime('%Y-%m-%d %H:%M:%S')} EST")
print("="*80)

if not recs.empty:
    # Formatting
    df_disp = recs.copy()
    df_disp['Confidence'] = df_disp['Confidence'].apply(lambda x: f"{x*100:.1f}%")
    df_disp['Exp_Return_%'] = df_disp['Exp_Return_%'].apply(lambda x: f"{x:.2f}%")
    df_disp['Exp_Dollar_Move'] = df_disp['Exp_Dollar_Move'].apply(lambda x: f"${x:.2f}")
    df_disp['Price'] = df_disp['Price'].apply(lambda x: f"${x:.2f}")
    df_disp['ADX'] = df_disp['ADX'].apply(lambda x: f"{x:.1f}")
    
    display(df_disp.sort_values(by='Action', ascending=False))

    print("\n" + "="*80)
    print("       TOP OPPORTUNITIES SUMMARY")
    print("="*80)
    
    # Best Picks
    top_up = recs.sort_values(by="Prob_Up", ascending=False).head(1)
    top_down = recs.sort_values(by="Prob_Down", ascending=False).head(1)
    top_gain = recs.sort_values(by="Exp_Return_%", ascending=False).head(1)
    
    print(f"\n[MOST LIKELY TO RISE]: {top_up['Ticker'].values[0]} ({top_up['Prob_Up'].values[0]*100:.1f}% Prob)")
    print(f"[MOST LIKELY TO FALL]: {top_down['Ticker'].values[0]} ({top_down['Prob_Down'].values[0]*100:.1f}% Prob)")
    print(f"\n[BIGGEST EXPECTED GAIN]: {top_gain['Ticker'].values[0]} (+{top_gain['Exp_Return_%'].values[0]:.2f}% | {top_gain['Exp_Dollar_Move'].values[0]:.2f}$)")
    
    # Market Context
    spy_row = recs[recs['Ticker'] == 'SPY']
    if not spy_row.empty:
        print(f"\n[MARKET CONTEXT (SPY)]: {spy_row['Action'].values[0]} | Trend Strength (ADX): {spy_row['ADX'].values[0]:.1f}")

print("\n" + "="*80)
print("Strategy: Buy at Market Open, Sell at Market Close.")
print("Calibration: XGBoost + Advanced Feature Set (VWAP, ADX, Vol-Z)")
print("="*80)


Processing 368 tickers with Advanced Calibration...



1 Failed download:
['MMC']: YFPricesMissingError('possibly delisted; no price data found  (period=5y) (Yahoo error = "No data found, symbol may be delisted")')

1 Failed download:
['SQ']: YFPricesMissingError('possibly delisted; no price data found  (period=5y) (Yahoo error = "No data found, symbol may be delisted")')

1 Failed download:
['WBA']: YFPricesMissingError('possibly delisted; no price data found  (period=5y) (Yahoo error = "No data found, symbol may be delisted")')



       STOCK PREDICTOR: CALIBRATED SIGNALS | 2026-06-01 11:45:22 EST


,Ticker,Action,Confidence,Exp_Return_%,Exp_Dollar_Move,Price,Prob_Up,Prob_Down,ADX,Vol_Regime
1,AAL,SELL/SHORT,55.7%,-0.76%,$-0.11,$14.26,0.315466,0.557131,34.8,High
294,ROKU,SELL/SHORT,64.5%,-0.21%,$-0.27,$129.40,0.175923,0.645131,21.8,Normal
330,TSM,SELL/SHORT,59.3%,0.06%,$0.26,$441.19,0.338764,0.593020,13.2,Normal
219,MDLZ,SELL/SHORT,57.4%,-0.60%,$-0.36,$59.99,0.196918,0.573825,22.7,Normal
224,MGM,SELL/SHORT,55.6%,0.14%,$0.07,$50.10,0.295282,0.556470,43.3,High
...,...,...,...,...,...,...,...,...,...,...
39,AVGO,BUY,63.5%,-0.12%,$-0.55,$458.00,0.634882,0.252485,13.7,Normal
35,ARM,BUY,62.4%,-1.01%,$-4.24,$418.71,0.623615,0.310713,41.9,High
34,ARE,BUY,56.3%,0.83%,$0.41,$49.22,0.562676,0.216891,45.2,Normal
33,APTV,BUY,57.9%,1.36%,$0.92,$67.77,0.579126,0.167282,47.0,Normal



       TOP OPPORTUNITIES SUMMARY

[MOST LIKELY TO RISE]: PANW (74.3% Prob)
[MOST LIKELY TO FALL]: VRTX (71.3% Prob)

[BIGGEST EXPECTED GAIN]: FDX (+4.00% | 13.34$)

[MARKET CONTEXT (SPY)]: HOLD | Trend Strength (ADX): 35.1

Strategy: Buy at Market Open, Sell at Market Close.
Calibration: XGBoost + Advanced Feature Set (VWAP, ADX, Vol-Z)
